In [1]:
# !rm -rf logs/ # clear logs
# !rm -rf optimizer_output/

# Imports

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [3]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine            import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools          import Nevergrad_Spice_Bode_Optimizer
from symxplorer.designer_tools.utils    import Frequency_Weight
from symxplorer.designer_tools.domains  import Project_Setup
from symxplorer.designer_tools.tf_models import Second_Order_BP_TF, cascade_tf

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-09-26 09:06:50,799 - matplotlib - matplotlib data path: /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data
2025-09-26 09:06:50,806 - matplotlib - CONFIGDIR=/headless/.config/matplotlib
2025-09-26 09:06:50,845 - matplotlib - interactive is False
2025-09-26 09:06:50,845 - matplotlib - platform is linux
2025-09-26 09:06:51,184 - matplotlib - CACHEDIR=/headless/.cache/matplotlib
2025-09-26 09:06:51,187 - matplotlib.font_manager - Using fontManager instance from /headless/.cache/matplotlib/fontlist-v390.json
2025-09-26 09:06:51,456 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-09-26 09:06:51,462 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

09:06:51 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
09:06:51 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-26_09-06-51.log
09:06:51 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [5]:
# s = sp.symbols("s")
# target_tf = (s + 1) / (s**2 + 24*s + 2)
fc=1e9
q=10
k_bp=1e3
filter_inst = Second_Order_BP_TF(q=q, fc=fc, k_bp=k_bp)
target_tf   = filter_inst.get_tf()
target_tf

200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)

In [6]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

09:06:51 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
09:06:51 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=100, random_seed=48
09:06:51 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
09:06:51 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
09:06:51 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
09:06:51 - SymXplorer.domains: [INFO] 	Number of target specs: 3
09:06:51 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=1e6, tolerance=10000.0, goal=GoalType.EXACT, sim_type=SimType.AC, enable=True)
09:06:51 - SymXplorer.domains: [INFO] 		- TargetSpec(name=q, target=10, tolerance=1, goal=GoalType.EXACT, sim_type=SimType.AC, enable=True)
09:06:51 - SymXplorer.domains: [INFO] 		- TargetSpec(name=gain, target=10, tole

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06), 'min_

In [7]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

09:06:51 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
09:06:51 - SymXplorer.spicelib: [INFO] --------------------------------------------------
09:06:51 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
09:06:51 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
09:06:51 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
09:06:51 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
09:06:51 - SymXplorer.spicelib: [INFO] --------------------------------------------------
09:06:51 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
09:06:51 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
09:06:51 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
09:06:51 - SymXplorer.spicelib: [INFO] Te

In [8]:
circuit_optimizer = Nevergrad_Spice_Bode_Optimizer(
    spicelib_wrapper=wrapper,
    target_tf=target_tf,
    output_node='vout',
    frequency_weight=Frequency_Weight(lower=fc/10, upper=fc*10),
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [9]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

09:06:51 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
09:06:51 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
09:06:52 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
09:06:52 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
09:06:52 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
09:06:52 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [10]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [11]:
circuit_optimizer.optimize()

09:06:52 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 100
Optimizing:   0%|          | 0/100 [00:00<?, ?trial/s]2025-09-26 09:06:52,153 - nevergrad.optimization.optimizerlib - CMA selected CMAbounded optimizer.
09:06:52 - SymXplorer.optimizer: [INFO] computing the target complex response for 200000000000.0*pi*s/(s**2 + 200000000.0*pi*s + 4.0e+18*pi**2)
Optimizing: 100%|██████████| 100/100 [00:33<00:00,  3.00trial/s]


[{'params': {'x_dut_nfet_w': 4.781102768876611,
   'x_dut_nfet_l': 46.19603019225511,
   'x_dut_cap_w': 40.807808776400016,
   'x_dut_cap_l': 43.50897660571927,
   'x_dut_res_s_l': 61.85137050082868,
   'x_dut_res_s_w': 44.33857366905921,
   'x_dut_res_3_l': 46.70847089713208,
   'x_dut_res_3_w': 48.52080130884472},
  'loss': np.float64(7569.347707579978)},
 {'params': {'x_dut_nfet_w': 18.173962929680105,
   'x_dut_nfet_l': 53.84656503564719,
   'x_dut_cap_w': 57.19379369170719,
   'x_dut_cap_l': 52.778672396378546,
   'x_dut_res_s_l': 42.85997191065726,
   'x_dut_res_s_w': 42.82447445510841,
   'x_dut_res_3_l': 45.111539847807045,
   'x_dut_res_3_w': 45.21473625452491},
  'loss': np.float64(90566.4414361841)},
 {'params': {'x_dut_nfet_w': 9.345048401207226,
   'x_dut_nfet_l': 54.763212505769395,
   'x_dut_cap_w': 52.37718656307508,
   'x_dut_cap_l': 40.05474299498853,
   'x_dut_res_s_l': 46.8858463763439,
   'x_dut_res_s_w': 48.98769013170926,
   'x_dut_res_3_l': 52.0547168187976,
   

In [12]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

09:07:26 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
09:07:26 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


In [13]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss = out

09:07:26 - SymXplorer.optimizer: [INFO] best loss: 1580.6685395580512


In [14]:
circuit_optimizer.plot_solution(best_param)

09:07:27 - SymXplorer.optimizer: [INFO] total loss: 1580.6685395580512
09:07:27 - SymXplorer.optimizer: [INFO] mag_loss 356.99073947981503, phase_loss 16032.945167120464
